# Notebook para predecir la cancelación de servicios de telecomunicaciones

En este notebook se realiza el desarrollo de un análisis desde la exploración de los datos hasta la construcción de un modelo que permita predecir la cancelación de un servicios de telecomunicaciones. Para esto se utilizan técnicas de análisis de datos, visualización y modelado predictivo. El conjunto de datos se extrajo de [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) y contiene información sobre los clientes de una empresa de telecomunicaciones.


_Richard Andrés M._

&nbsp;

---

## Preparación del entorno

### Librerías

Se importan todas las librerías necesarias para el análsis.

In [ ]:
# Librerías para el procesamiento de los datos
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Librerías para el modelado predictivo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

# Manejo de warnings
import warnings
warnings.filterwarnings('ignore')

### Configuración

Se definen variables de configuración y rutas del proyecto.

In [ ]:
# Rutas
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'

RAW_PATH = DATA_DIR / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
INTERIM_PATH = DATA_DIR / 'interim' / 'telco_interim.csv'
PROCESSED_PATH = DATA_DIR / 'processed' / 'telco_processed.csv'


# Colores
# Verde = No Churn
# Rojo = Churn
PALETTE = ['#2ecc71', '#e74c3c']

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ── Rutas del proyecto ──────────────────────────────────────────────────────

&nbsp;

---

## Proceso de ETL

En esta sección se realiza la extracción, transformación y carga de los datos. Se cargan los datos desde un archivo CSV, se limpian y se preparan para el análisis.

&nbsp;

---
### Exploración inicial del conjunto de datos

En esta sección se carga el conjunto de datos y se realiza la exploración de su estructura. El conjunto de datos utilizado cuenta con la siguientes columnas:
- **customerID**: Identificador único de cada cliente.
- **gender**: Género del cliente (Masculino o Femenino).
- **SeniorCitizen**: Indica si el cliente es una persona mayor (1) o no (0).
- **Partner**: Indica si el cliente tiene pareja (Yes o No).
- **Dependents**: Indica si el cliente tiene dependientes (Yes o No).
- **tenure**: Número de meses que el cliente ha estado con la empresa.
- **PhoneService**: Indica si el cliente tiene servicio telefónico (Yes o No).
- **MultipleLines**: Indica si el cliente tiene múltiples líneas telefónicas (Yes, No, No phone service).
- **InternetService**: Tipo de servicio de internet contratado por el cliente (DSL, Fiber optic, No).
- **OnlineSecurity**: Indica si el cliente tiene seguridad en línea (Yes, No, No internet service).
- **OnlineBackup**: Indica si el cliente tiene respaldo en línea (Yes, No, No internet service).
- **DeviceProtection**: Indica si el cliente tiene protección de dispositivos (Yes, No, No internet service).
- **TechSupport**: Indica si el cliente tiene soporte técnico (Yes, No, No internet service).
- **StreamingTV**: Indica si el cliente tiene servicio de streaming de televisión (Yes, No, No internet service).
- **StreamingMovies**: Indica si el cliente tiene servicio de streaming de películas (Yes, No, No internet service).
- **Contract**: Tipo de contrato del cliente (Month-to-month, One year, Two year).
- **PaperlessBilling**: Indica si el cliente tiene facturación sin papel (Yes o No).
- **PaymentMethod**: Método de pago utilizado por el cliente (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic)).
- **MonthlyCharges**: Cargos mensuales que el cliente paga por los servicios.
- **TotalCharges**: Cargos totales que el cliente ha pagado por los servicios.
- **Churn**: Indica si el cliente ha cancelado el servicio (Yes o No).

In [ ]:
# Carga de datos
df_raw = pl.read_csv(RAW_PATH)

df_raw.shape

In [ ]:
df_raw.head(5)

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df_raw.select(pl.col(pl.NUMERIC_DTYPES)).describe()

In [ ]:
df_raw.null_count()

In [ ]:
null_df = pl.DataFrame({
    'columna':   df_raw.columns,
    'nulos':     [df_raw[c].null_count() for c in df_raw.columns],
    'pct_nulos': [
        round(df_raw[c].null_count() / df_raw.shape[0] * 100, 2)
        for c in df_raw.columns
    ]
}).filter(pl.col('nulos') > 0)

null_df

Solo la columna "TotalCharges" tiene valores nulos, representando el 16% de los registros.

In [ ]:
df_raw.filter(
    pl.col('TotalCharges').is_null()).select(pl.col(pl.NUMERIC_DTYPES)
).describe()

In [ ]:
df_raw.filter(
    pl.col('TotalCharges').is_null()).select(pl.col(pl.Utf8)
)

> Se identifica que los valores nulos de "TotalCharges" corresponden a clientes con una duración de contrato de 0 meses, lo que sugiere que estos clientes no han pagado ningún cargo total debido a que no han utilizado los servicios probablemente porque son nuevos clientes.

In [ ]:
# Distribución de la variable objetivo (Churn)
churn_no  = (df_raw['Churn'] == 'No').sum()
churn_yes = (df_raw['Churn'] == 'Yes').sum()
total     = df_raw.shape[0]

labels = ['No Churn', 'Churn']
values = [churn_no, churn_yes]
colors = PALETTE

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribución de la Variable Objetivo - Churn', fontsize=14, fontweight='bold', y=1.01)

# Gráfico de barras
bars = axes[0].bar(
    labels, values, color=colors, width=0.45, edgecolor='white', linewidth=1.5
)

for bar, val in zip(bars, values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 60,
        f'{val:,}\n({val/total*100:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
axes[0].set_ylim(0, max(values) * 1.2)
axes[0].set_title('Conteo por clase', fontsize=12)
axes[0].set_ylabel('Número de clientes')

# Gráfico de torta
wedges, texts, autotexts = axes[1].pie(
    values, labels=labels, autopct='%1.1f%%',
    startangle=90, colors=colors,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('Proporción relativa', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de las variables categóricas más relevantes en relación con el Churn
cat_vars = {
    'Contract': 'Tipo de contrato',
    'InternetService': 'Servicio de Internet',
    'PaymentMethod': 'Método de pago',
    'TechSupport': 'Soporte técnico',
}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Tasa de Churn por Variable Categórica', fontsize=15, fontweight='bold')

for ax, (col, title) in zip(axes.flatten(), cat_vars.items()):
    categories = df_raw[col].unique().to_list()
    churn_rates = []
    counts_list = []
    
    for cat in sorted(categories):
        subset = df_raw.filter(pl.col(col) == cat)
        rate = (subset['Churn'] == 'Yes').sum() / subset.shape[0] * 100
        churn_rates.append(rate)
        counts_list.append(subset.shape[0])

    sorted_cats = sorted(categories)
    bars = ax.bar(range(len(sorted_cats)), churn_rates, color='#e74c3c', alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(sorted_cats)))
    ax.set_xticklabels(sorted_cats, rotation=15, ha='right', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Tasa de Churn (%)')
    ax.set_ylim(0, max(churn_rates) * 1.3)

    for bar, rate, count in zip(bars, churn_rates, counts_list):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{rate:.1f}%\n(n={count:,})',
            ha='center', va='bottom', fontsize=8
        )

plt.tight_layout()
plt.show()

> Al revisar las variables categóricas, se observa que la mayoría de los clientes tienen contratos mensuales, utilizan servicios de internet por fibra óptica y pagan mediante cheques electrónicos.

&nbsp;

---

### Limpieza de Datos

Esta primera fase de transformación se centra en garantizar la **calidad de los datos** antes de cualquier análisis estadístico o modelado.

In [ ]:
# Eliminación de identificar único del cliente
df = df_raw.drop('customerID')

# Eliminación de registros con TotalCharges nulo
df = df.drop_nulls(subset=['TotalCharges'])

print(f'\nDimensiones tras la limpieza: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Registros eliminados: {df_raw.shape[0] - df.shape[0]}')

In [ ]:
# Análisis de outliers mediante el método IQR
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Visualización: boxplots de las tres variables numéricas
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Distribución de Variables Numéricas — Dataset Limpio', fontsize=13, fontweight='bold')

for ax, col in zip(axes, numeric_cols):
    data = df[col].to_numpy()
    ax.boxplot(data, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Valor')

plt.tight_layout()
plt.show()

> No se identifican valores atípicos en las variables numéricas.

In [ ]:
# Guardar dataset limpio en data/interim/
df_interim = df.clone()
df_interim.write_csv(INTERIM_PATH)

print(f'✓ Dataset limpio guardado en: {INTERIM_PATH}')

&nbsp;


### Análisis Exploratorio Visual

Antes de transformar los datos, se realiza un análisis visual que permite:

- Identificar la forma de las distribuciones (sesgos, bimodalidad).
- Detectar diferencias en el comportamiento de las variables según el churn.
- Comprender las relaciones entre variables (multicolinealidad potencial).


In [ ]:
# Distribución de variables numéricas segmentada por Churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Distribución de variables numéricas según Churn', fontsize=14, fontweight='bold'
)

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    for label, color in zip(['No', 'Yes'], PALETTE):
        vals = df_interim.filter(pl.col('Churn') == label)[col].to_numpy()
        ax.hist(
            vals, bins=35, alpha=0.6, color=color, density=True, label=f'Churn={label}'
        )
        
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Densidad')
    ax.legend()

plt.tight_layout()
plt.show()

> Se identifica que clientes con un menor tiempo de contrato (tenure) tienen una mayor tasa de cancelación (churn), lo que sugiere que los clientes nuevos son más propensos a cancelar el servicio. Además, se observa que los clientes con contratos mensuales tienen una tasa de cancelación significativamente más alta que aquellos con contratos anuales o bienales.

In [ ]:
# Tasa de churn para variables categóricas adicionales
more_cat = {
    'SeniorCitizen': 'Cliente es persona mayor (0/1)',
    'Partner': 'Tiene pareja',
    'Dependents': 'Tiene dependientes',
    'MultipleLines': 'Múltiples líneas',
    'StreamingTV': 'Streaming TV',
    'PaperlessBilling': 'Factura digital'
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    'Tasa de Churn por variables demográficas y de servicio', fontsize=14, fontweight='bold'
)

for ax, (col, title) in zip(axes.flatten(), more_cat.items()):
    categories = sorted(df_interim[col].unique().cast(pl.String).to_list())
    rates  = []
    counts = []

    for cat in categories:
        sub = df_interim.filter(pl.col(col).cast(pl.String) == cat)
        rate = (sub['Churn'] == 'Yes').sum() / sub.shape[0] * 100
        rates.append(rate)
        counts.append(sub.shape[0])

    bars = ax.bar(categories, rates, color='#e74c3c', alpha=0.8, edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Tasa de Churn (%)')
    ax.set_ylim(0, max(rates) * 1.35 + 1)
    for bar, r, n in zip(bars, rates, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{r:.1f}%\n(n={n:,})', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

> Teniendo en cuenta las variables categóticas, se identifica una mayor tasa de cancelación en los siguientes escenarios:
> - Clientes mayores.
> - Clientes sin pareja.
> - Clientes sin dependientes.
> - Clientes con múltiples líneas telefónicas.
> - Clientes que no cuentan con servicio de Streaming.
> - Clientes que cuentan con factura digital.

In [ ]:
# Mapa de correlación entre variables numéricas
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
corr_matrix  = np.corrcoef([df_interim[c].to_numpy() for c in numeric_cols])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Correlación de Pearson')

ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=30, ha='right')
ax.set_yticklabels(numeric_cols)
ax.set_title(
    'Matriz de correlación — Variables numéricas', fontsize=13, fontweight='bold'
)

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(
            j, i, f'{corr_matrix[i, j]:.2f}',
            ha='center', va='center', fontsize=12, fontweight='bold',
            color='black'
        )

plt.tight_layout()
plt.show()

print('\n► tenure y TotalCharges tienen alta correlación (~0.83).')
print('  Esto es esperable: a más tiempo de cliente, mayor cargo acumulado.')
print('  Se mantendrán ambas variables por su signficado diferente.')
print('  El StandardScaler aplicado en la sección 5 no elimina la colinealidad;')
print('  los modelos de árbol son robustos ante esto y la regresión usa regularización.')

> Se identifico que `tenure` y `TotalCharges` tienen una correlación moderada, esto es lógico ya que a medida que un cliente permanece más tiempo con la empresa, el cargo acumulado aumenta. Aunque existe una correlación entre estas variables, inicialmente se mantendrán estas variables ya que cada una posee significados diferentes. Por otro lado, se identifica que entre `MonthlyChargers` y `TotalCharges` también una correlación indicando que a medida que el cargo acumulado aumenta, el cargo mensual también aumenta indicando que a medida que pasa el tiempo es probable que la tarifa aumente elevando el cargo total acumulado.

&nbsp;

---

### Transformaciones y Normalización

Los algoritmos de Machine Learning requieren que todas las variables estén expresadas en formato numérico y, para ciertos modelos, en escalas comparables. Esta sección aplica las siguientes transformaciones:

- Codificación de variables categóricas.
- Discretización de `tenure`.
- Normalización.

In [ ]:
df_enc = df_interim.clone()

# Codificación de variables binarias: si o no
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
df_enc = df_enc.with_columns([
    pl.when(pl.col(c) == 'Yes').then(pl.lit(1)).otherwise(pl.lit(0)).cast(pl.Int8).alias(c)
    for c in yes_no_cols
])

# Códificación de variables binarias: hombre o mujer
df_enc = df_enc.with_columns(
    pl.when(pl.col('gender') == 'Male').then(pl.lit(1)).otherwise(pl.lit(0))
    .cast(pl.Int8).alias('gender')
)

df_enc.select(yes_no_cols + ['gender']).head(5)

In [ ]:
# Codificación de variables con tres categorías
three_cat_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df_enc = df_enc.to_dummies(columns=three_cat_cols, separator='_', drop_first=True)

cols_after = [
    x for x in df_enc.columns if any(x.startswith(p) for p in three_cat_cols)
]
df_enc.select(cols_after).head(5)

In [ ]:
# Códificación ordinal para Contract
df_enc = df_enc.with_columns(
    pl.when(pl.col('Contract') == 'Month-to-month').then(pl.lit(0))
    .when(pl.col('Contract') == 'One year').then(pl.lit(1))
    .otherwise(pl.lit(2))
    .cast(pl.Int8).alias('Contract')
)

# Códificación para InternetService y PaymentMethod
df_enc = df_enc.to_dummies(
    columns=['InternetService', 'PaymentMethod'], drop_first=True
)

# Limpiar nombres de columna (eliminar espacios y paréntesis)
rename_map = {
    c: c.replace(' ', '_').replace('(', '_').replace(')', '_')
    for c in df_enc.columns
}
df_enc = df_enc.rename(rename_map)

# Codificación del target
df_enc = df_enc.with_columns(
    pl.when(pl.col('Churn') == 'Yes').then(pl.lit(1)).otherwise(pl.lit(0))
    .cast(pl.Int8).alias('Churn')
)

In [ ]:
# Discretización de la variable tenure
def assign_tenure_group(tenure: int) -> str:
    if tenure <= 12:
        return 'Nuevo (0-12m)'
    elif tenure <= 36:
        return 'En desarrollo (13-36m)'
    elif tenure <= 60:
        return 'Establecido (37-60m)'
    else:
        return 'Leal (>60m)'

df_enc = df_enc.with_columns(
    pl.col('tenure').map_elements(
        assign_tenure_group, return_dtype=pl.String
    ).alias('tenure_group')
)

# Distribución de los grupos
group_counts = df_enc['tenure_group'].value_counts().sort('count', descending=True)
churn_by_group = []

for grp in group_counts['tenure_group'].to_list():
    sub  = df_enc.filter(pl.col('tenure_group') == grp)
    rate = sub['Churn'].mean() * 100
    churn_by_group.append(rate)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    group_counts['tenure_group'].to_list(),
    group_counts['count'].to_list(),
    color=['#e74c3c', '#e67e22', '#3498db', '#2ecc71'],
    edgecolor='white'
)
ax2 = ax.twinx()
ax2.plot(
    range(len(churn_by_group)), churn_by_group, 'ko--', linewidth=2, markersize=8, label='Tasa churn (%)'
)
ax2.set_ylabel('Tasa de Churn (%)', color='black')
ax2.set_ylim(0, 70)

ax.set_xlabel('Grupo de Antigüedad')
ax.set_ylabel('Número de Clientes')
ax.set_title('Distribución por grupos de antigüedad y tasa de Churn', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right')

for bar, count in zip(bars, group_counts['count'].to_list()):
    ax.text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
        f'{count:,}', ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.show()

> Los clientes nuevos (tenure 0-12 meses) tienen una tasa de cancelación significativamente más alta que los clientes con mayor tiempo de contrato, lo que sugiere que la duración del contrato es un factor importante en la retención de clientes.

In [ ]:
# Eliminar la columna de grupos
df_processed = df_enc.drop('tenure_group')

# Verificar que todas las columnas sean numéricas
non_numeric = [
    col
    for col, dt in zip(df_processed.columns, df_processed.dtypes)
    if dt == pl.String
]

if non_numeric:
    print(f'⚠ Columnas no numéricas encontradas: {non_numeric}')
else:
    print('✓ Todas las columnas son numéricas.')

# Guardar el dataset procesado
df_processed.write_csv(PROCESSED_PATH)
print(f'\n✓ Dataset procesado guardado en: {PROCESSED_PATH}')

&nbsp;

---

## Reducción de Dimensionalidad — Análisis de Componentes Principales (PCA)

El **PCA** es una técnica de reducción de dimensionalidad no supervisada que transforma el conjunto de variables en un nuevo espacio de menor dimensión, donde las nuevas variables (*componentes principales*) son combinaciones lineales ortogonales de las originales y están ordenadas por la varianza que explican.

In [ ]:
# Preparar matrices para PCA
feature_cols = [c for c in df_processed.columns if c != 'Churn']
X_all = df_processed.select(feature_cols).to_numpy().astype(float)
y_all = df_processed['Churn'].to_numpy()

# Estandarización
scaler_pca = StandardScaler()
X_scaled   = scaler_pca.fit_transform(X_all)

# PCA
pca_full  = PCA().fit(X_scaled)
cumvar    = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_comp_90 = np.argmax(cumvar >= 90) + 1

# PCA 2D
pca_2d  = PCA(n_components=2)
X_pca2d = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análisis de Componentes Principales (PCA)', fontsize=14, fontweight='bold')


for label, color, name in zip([0, 1], PALETTE, ['No Churn', 'Churn']):
    mask = y_all == label
    axes[0].scatter(
        X_pca2d[mask, 0], X_pca2d[mask, 1],
        alpha=0.25, s=8, color=color, label=name
    )

axes[0].set_xlabel(f'PC1  ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[0].set_ylabel(f'PC2  ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[0].set_title('Proyección 2D', fontsize=11)
axes[0].legend(markerscale=4, fontsize=11)


axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'b-o', markersize=4)
axes[1].axhline(90, color='#e74c3c', linestyle='--', label=f'90% varianza ({n_comp_90} componentes)')
axes[1].axhline(95, color='#e67e22', linestyle='--', label=f'95% varianza')
axes[1].fill_between(range(1, len(cumvar) + 1), cumvar, alpha=0.1, color='blue')
axes[1].set_xlabel('Número de componentes principales')
axes[1].set_ylabel('Varianza explicada acumulada (%)')
axes[1].set_title('Scree Plot — Varianza acumulada por componente', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> Se identifica que los primeros dos componentes principales explican aproximadamente el 50% de la varianza total, lo que indica que una gran parte de la información se puede representar en un espacio bidimensional. Sin embargo, dado que las clases de churn están superpuestas en la proyección 2D, esto sugiere que un modelo lineal podría no ser suficiente para separar las clases, y se requeriría un modelo no lineal para capturar las complejidades de los datos.

&nbsp;

---

## Modelado Predictivo

Se entrenan tres modelos de clasificación supervisada, cada uno con un enfoque distinto:

| # | Modelo | Tipo | Fortalezas |
|---|--------|------|------------|
| 1 | **Regresión Logística** | Paramétrico lineal | Alta interpretabilidad, baseline sólido, probabilidades calibradas |
| 2 | **Random Forest** | Ensemble (bagging) | Maneja no linealidades, robusto a outliers, importancia de variables |
| 3 | **Gradient Boosting** | Ensemble (boosting) | Generalmente mejor desempeño, captura patrones complejos |

### Manejo del desbalance de clases con SMOTE

Con una tasa de churn de ~27%, los modelos pueden aprender a predecir siempre la clase mayoritaria y aún así alcanzar alta accuracy (problema de la "accuracy paradox").

**SMOTE** (*Synthetic Minority Over-sampling Technique*) genera ejemplos sintéticos de la clase minoritaria interpolando entre vecinos cercanos en el espacio de features. Se aplica **sólo sobre los datos de entrenamiento** para evitar data leakage.

In [ ]:
# Cargar el dataset procesado desde disco (simula el inicio del pipeline de modelado)
df_model = pl.read_csv(PROCESSED_PATH)

FEATURE_COLS = [c for c in df_model.columns if c != 'Churn']
TARGET_COL   = 'Churn'

X = df_model.select(FEATURE_COLS).to_numpy().astype(float)
y = df_model[TARGET_COL].to_numpy()

# Conjuntos de entramiento y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=300, stratify=y
)

print(f'{"Conjunto":<12} {"Registros":>10} {"Churn (%)":>12}')
print('-' * 36)
for name, arr_y in [('Entrenamiento', y_train), ('Test', y_test)]:
    ch_pct = arr_y.sum() / len(arr_y) * 100
    print(f'{name:<12} {len(arr_y):>10,} {ch_pct:>11.1f}%')

# Estandarización de los datos
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Ajuste de balanceo con SMOTE sobre datos de entrenamiento ya escalados
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print(f'\nDespués de SMOTE (train):')
for cls in [0, 1]:
    n = (y_train_sm == cls).sum()
    print(f'  Churn={cls}: {n:,} ({n/len(y_train_sm)*100:.1f}%)')

In [ ]:
# Centraliza el cálculo de métricas para garantizar comparación homogénea entre modelos
def evaluate_model(name, model, X_tr, y_tr, X_ts, y_ts) -> tuple[dict, object, np.ndarray, np.ndarray]:
    """
    Entrena el modelo, evalúa en test y retorna un dict de métricas
    junto con las predicciones y probabilidades.
    """
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_ts)
    y_prob = model.predict_proba(X_ts)[:, 1]

    metrics = {
        'Modelo': name,
        'Accuracy': round(accuracy_score(y_ts, y_pred), 4),
        'Precision': round(precision_score(y_ts, y_pred), 4),
        'Recall': round(recall_score(y_ts, y_pred), 4),
        'F1-Score': round(f1_score(y_ts, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_ts, y_prob), 4)
    }

    print(f'\n{"="*58}')
    print(f'  {name}')
    print(f'{"="*58}')
    print(classification_report(y_ts, y_pred, target_names=['No Churn', 'Churn']))

    return metrics, model, y_pred, y_prob

In [ ]:
# Modelo 1: Regresión Logística
lr_model = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
results = {}

metrics_lr, m_lr, pred_lr, prob_lr = evaluate_model(
    'Regresión Logística', lr_model, X_train_sm, y_train_sm, X_test_sc, y_test
)
results['Regresión Logística'] = (metrics_lr, m_lr, pred_lr, prob_lr)

In [ ]:
# Modelo 2: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

metrics_rf, m_rf, pred_rf, prob_rf = evaluate_model(
    'Random Forest', rf_model, X_train_sm, y_train_sm, X_test_sc, y_test
)
results['Random Forest'] = (metrics_rf, m_rf, pred_rf, prob_rf)

In [ ]:
# Modelo 3: Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

metrics_gb, m_gb, pred_gb, prob_gb = evaluate_model(
    'Gradient Boosting', gb_model, X_train_sm, y_train_sm, X_test_sc, y_test
)
results['Gradient Boosting'] = (metrics_gb, m_gb, pred_gb, prob_gb)

### Comparación y Selección del Modelo Final


| Métrica | Fórmula | Interpretación en contexto de Churn |
|---------|---------|--------------------------------------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | % de predicciones correctas totales |
| **Precision** | TP/(TP+FP) | De los clientes predichos como churn, ¿cuántos realmente cancelaron? |
| **Recall** | TP/(TP+FN) | De los clientes que sí cancelaron, ¿cuántos detectamos? |
| **F1-Score** | 2·(P·R)/(P+R) | Media armónica de Precision y Recall (balance entre ambas) |
| **ROC-AUC** | Área bajo curva ROC | Capacidad de separar las dos clases, independiente del umbral |

> **Métrica prioritaria:** En predicción de churn es preferible maximizar el **Recall** (minimizar Falsos Negativos), ya que retener a un cliente que iba a cancelar es más valioso que evitar una retención innecesaria. El **F1-Score** y el **ROC-AUC** se usan como indicadores complementarios.

### Comparativo de métricas

In [ ]:
all_metrics = [m for m, *_ in results.values()]

# Gráfico de barras agrupadas
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names  = list(results.keys())
n_models     = len(model_names)
model_colors = ['#3498db', '#2ecc71', '#e74c3c']

x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(15, 6))
for i, (name, color) in enumerate(zip(model_names, model_colors)):
    vals = [results[name][0][m] for m in metric_names]
    bars = ax.bar(
        x + i * width, vals, width, label=name, color=color,
        alpha=0.85, edgecolor='white'
    )

    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.004,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8, rotation=0)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Valor de la métrica', fontsize=11)
ax.set_title('Comparación de Métricas entre los Tres Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(0.80, color='gray', linestyle=':', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC comparativas
fig, ax = plt.subplots(figsize=(9, 7))

for (name, (metrics, model, pred, prob)), color in zip(results.items(), model_colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = metrics['ROC-AUC']
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name}  (AUC = {auc:.3f})')

# Línea de clasificador aleatorio
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Clasificador aleatorio (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

ax.set_xlabel('Tasa de Falsos Positivos  (1 – Especificidad)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos  (Recall / Sensibilidad)', fontsize=12)
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Matrices de Confusión — Conjunto de Test', fontsize=14, fontweight='bold')

for ax, (name, (metrics, model, pred, prob)) in zip(axes, results.items()):
    cm   = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicción', fontsize=10)
    ax.set_ylabel('Valor real', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Selección del modelo definitivo e Importancia de Variables
best_name = max(results.keys(), key=lambda k: results[k][0]['Recall'])
best_metrics, best_model, best_pred, best_prob = results[best_name]

print(f'Modelo seleccionado: {best_name}')
print(f'\n Métricas en test:')
for metric, val in best_metrics.items():
    if metric != 'Modelo':
        print(f'     {metric:<12} {val:.4f}')

# Importancia de variables
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    # Para Regresión Logística se usan los valores absolutos de los coeficientes
    importances = np.abs(best_model.coef_[0])

importance_df = pl.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': importances.tolist(),
}).sort('importance', descending=True)

top_n  = 15
top_df = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top_df['feature'].to_list()[::-1],
    top_df['importance'].to_list()[::-1],
    color='#3498db', alpha=0.85, edgecolor='white'
)
ax.set_xlabel('Importancia relativa', fontsize=12)
ax.set_title(f'Top {top_n} Variables más Predictivas — {best_name}',
             fontsize=13, fontweight='bold')

# Anotar valores
for bar, val in zip(bars, top_df['importance'].to_list()[::-1]):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

> Las variables que mayor incluencia tienen en el modelo es `tenure` y `TotalCharges` que pueden estar relacionados con la antigüedad de los clientes, además `TotalCharges` puede estar relacionado a que con un costo más alto puede generar insatisfacción; por otro lado, `Contract` que es el tiempo del contrato.

### Modelo aplicado

Se presentan tres perfiles de clientes hipotéticos que representan escenarios de riesgo diferenciado. Cada perfil es construido manualmente con características realistas y luego procesado con el mismo escalador del entrenamiento para obtener la predicción del modelo.

| Perfil | Características clave |
|--------|----------------------|
| **Cliente A — Alto riesgo** | Nuevo (2 meses), contrato mensual, fibra óptica, sin servicios de valor agregado |
| **Cliente B — Bajo riesgo** | Leal (70 meses), contrato bianual, DSL, servicios completos, débito automático |
| **Cliente C — Riesgo moderado** | Adulto mayor (18 meses), contrato anual, fibra óptica, solo soporte técnico |

In [ ]:
perfiles_info = [
    {
        'nombre': 'Cliente A — Alto riesgo',
        'descripcion': 'Nuevo (2 meses) · Contrato mensual · Fibra óptica · Sin seguridad online · Cheque electrónico',
        'vector': [
            0,       # gender: Femenino
            0,       # SeniorCitizen: No
            0,       # Partner: No
            0,       # Dependents: No
            2,       # tenure: 2 meses
            1,       # PhoneService: Sí
            1, 0,    # MultipleLines: No → (_No=1, _Yes=0)
            1, 0,    # InternetService: Fibra óptica → (_Fiber_optic=1, _No=0)
            0, 0,    # OnlineSecurity: No (categoría ref.) → ambos 0
            1, 0,    # OnlineBackup: No → (_No=1, _No_internet_service=0)
            0, 0,    # DeviceProtection: No (categoría ref.) → ambos 0
            0, 0,    # TechSupport: No (categoría ref.) → ambos 0
            0, 1,    # StreamingTV: Sí → (_No_internet_service=0, _Yes=1)
            0, 1,    # StreamingMovies: Sí → (_No_internet_service=0, _Yes=1)
            0,       # Contract: mes a mes
            1,       # PaperlessBilling: Sí
            0, 0, 0, # PaymentMethod: Cheque electrónico (categoría ref.)
            90.0,    # MonthlyCharges
            180.0,   # TotalCharges
        ],
    },
    {
        'nombre': 'Cliente B — Bajo riesgo',
        'descripcion': 'Leal (70 meses) · Contrato bianual · DSL · Todos los servicios · Débito automático',
        'vector': [
            1,       # gender: Masculino
            0,       # SeniorCitizen: No
            1,       # Partner: Sí
            1,       # Dependents: Sí
            70,      # tenure: 70 meses
            1,       # PhoneService: Sí
            0, 1,    # MultipleLines: Sí → (_No=0, _Yes=1)
            0, 0,    # InternetService: DSL (categoría ref.) → ambos 0
            0, 1,    # OnlineSecurity: Sí → (_No_internet_service=0, _Yes=1)
            0, 0,    # OnlineBackup: Sí (categoría ref.) → ambos 0
            0, 1,    # DeviceProtection: Sí → (_No_internet_service=0, _Yes=1)
            0, 1,    # TechSupport: Sí → (_No_internet_service=0, _Yes=1)
            0, 0,    # StreamingTV: No (categoría ref.) → ambos 0
            0, 0,    # StreamingMovies: No (categoría ref.) → ambos 0
            2,       # Contract: bianual
            0,       # PaperlessBilling: No
            1, 0, 0, # PaymentMethod: Débito bancario automático
            55.0,    # MonthlyCharges
            3850.0,  # TotalCharges
        ],
    },
    {
        'nombre': 'Cliente C — Riesgo moderado',
        'descripcion': 'Adulto mayor (18 meses) · Contrato anual · Fibra óptica · Solo soporte técnico · Tarjeta de crédito',
        'vector': [
            0,       # gender: Femenino
            1,       # SeniorCitizen: Sí
            0,       # Partner: No
            0,       # Dependents: No
            18,      # tenure: 18 meses
            1,       # PhoneService: Sí
            1, 0,    # MultipleLines: No → (_No=1, _Yes=0)
            1, 0,    # InternetService: Fibra óptica → (_Fiber_optic=1, _No=0)
            0, 0,    # OnlineSecurity: No (categoría ref.) → ambos 0
            0, 0,    # OnlineBackup: Sí (categoría ref.) → ambos 0
            0, 0,    # DeviceProtection: No (categoría ref.) → ambos 0
            0, 1,    # TechSupport: Sí → (_No_internet_service=0, _Yes=1)
            0, 0,    # StreamingTV: No (categoría ref.) → ambos 0
            0, 0,    # StreamingMovies: No (categoría ref.) → ambos 0
            1,       # Contract: anual
            1,       # PaperlessBilling: Sí
            0, 1, 0, # PaymentMethod: Tarjeta de crédito automática
            75.0,    # MonthlyCharges
            1350.0,  # TotalCharges
        ],
    },
]

# Construir matriz y escalar con el mismo scaler del entrenamiento
X_ej    = np.array([p['vector'] for p in perfiles_info])
X_ej_sc = scaler.transform(X_ej)

pred_ej = best_model.predict(X_ej_sc)
prob_ej = best_model.predict_proba(X_ej_sc)[:, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

nombres  = [p['nombre'] for p in perfiles_info]
colores  = ['#e74c3c' if pr >= 0.5 else ('#e67e22' if pr >= 0.3 else '#2ecc71')
            for pr in prob_ej]

barras = ax.barh(
    nombres[::-1],
    [p * 100 for p in prob_ej[::-1]],
    color=colores[::-1],
    height=0.5,
    edgecolor='white',
)

ax.axvline(50, color='#c0392b', linestyle='--', linewidth=1.5, label='Umbral decisión (50%)')
ax.set_xlabel('Probabilidad de Churn (%)', fontsize=12)
ax.set_title(f'Probabilidad de Churn por Perfil de Cliente — {best_name}',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 100)
ax.legend(fontsize=10)

for barra, prob in zip(barras, prob_ej[::-1]):
    ax.text(
        barra.get_width() + 1.5,
        barra.get_y() + barra.get_height() / 2,
        f'{prob:.1%}',
        va='center', fontsize=11, fontweight='bold',
    )

plt.tight_layout()
plt.show()

&nbsp;

---

## Conclusiones

### Hallazgos principales del análisis exploratorio

1. **Antigüedad del cliente (`tenure`):** Es el predictor más fuerte. Los clientes con menos de 12 meses tienen una tasa de churn superior al 45%.
2. **Tipo de contrato:** Los contratos mes a mes presentan ~4× más churn que los bianuales. Las campañas de conversión a contratos anuales tienen alto impacto de retención.
3. **Servicio de Internet de fibra óptica:** A pesar de ser el servicio premium, sus clientes muestran mayor churn —posiblemente por expectativas de precio/calidad no satisfechas.
4. **Soporte técnico y seguridad online:** Los clientes sin estos servicios tienen mayor propensión al abandono.

---

### Comparación de modelos

Los tres modelos lograron superar el 80% de ROC-AUC, lo cual indica buena capacidad discriminante. El modelo de **Regresión Logística** obtuvo el mejor `Recall` lo que lo convierte en un candidato sólido para despliegue, especialmente si se prioriza la retención de clientes.

---

### Recomendaciones de negocio

- **Intervención temprana:** Diseñar campañas de retención para clientes con menos de 6 meses de antigüedad.
- **Incentivos de contrato:** Ofrecer descuentos para migrar de contrato mensual a anual.
- **Monitoreo continuo:** Reentrenar el modelo trimestralmente con datos actualizados para mantener su desempeño.

